

---

# Dimensioneren M2



---
 

In [1]:
# import statements
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

import MechDesign.Helpers as HM

from MechDesign.Units.Units import m_, mm_, kg_, s_, N_, rpm_, W_, deg_
import MechDesign.Units.UnitMethods as UM

import MechDesign.RnM as RnM

## a. Uitgaande arm

In [2]:
# creating the chain to be calculated
CH = RnM.Chain()    

# gegevens
delta_theta = np.pi # [rad]
t_max = 2 # [s]
ro_staal = 7850 # [kg/m^3]
hoogte_uitgaandearm = 0.1 # [m] # aanname # volledige hoogte, dus holte en dikke wand samen
breedte_uitgaandearm = 0.1 # [m] # aanname # volledige breedte, dus holte en dikke wand samen
lengte_uitgaandearmx2 = 0.46 # [m]
lengte_uitgaandearmx1 = 0.14 # [m] # aanname 
massa_last = 8 # [kg]
t_dikte_balk = 0.015 # [m] # aanname rekening houdend met doorgang kabels

# berekeningen 
alpha_max = 4.5 * delta_theta / (t_max ** 2) # [rad/s^2] # verondersteld trapezoïdaal profiel, dus 4.5 keer de gemiddelde versnelling
alpha_gem = delta_theta / (t_max ** 2) # [rad/s^2]
omega_max = (delta_theta / t_max) * 1.5 # [rad/s] #verondersteld trapezoïdaal profiel, dus 1.5 keer de gemiddelde snelheid
omega_gem = delta_theta / t_max # [rad/s]
print(f"alpha_max: {alpha_max:.2f} rad/s^2")
print(f"omega_max: {omega_max:.2f} rad/s")
print(f"alpha_gem: {alpha_gem:.2f} rad/s^2")
print(f"omega_gem: {omega_gem:.2f} rad/s")
# berekening van de massa van de holle uitgaande as met vierkant profiel
oppervlakte_vierkantholprofiel = (breedte_uitgaandearm * hoogte_uitgaandearm) - ((breedte_uitgaandearm - 2 * t_dikte_balk) * (hoogte_uitgaandearm - 2 * t_dikte_balk))
volume_uitgaandearm = oppervlakte_vierkantholprofiel * (lengte_uitgaandearmx2 + lengte_uitgaandearmx1) # [m^3]
massa_uitgaandearm = volume_uitgaandearm * ro_staal # [kg]
print(f"massa uitgaande arm: {massa_uitgaandearm:.2f} kg")

# berekening van het traagheidsmoment van de holle uitgaande as met vierkant profiel
I_uitgaandearm = ((breedte_uitgaandearm * hoogte_uitgaandearm**3) - ((breedte_uitgaandearm - 2 * t_dikte_balk) * (hoogte_uitgaandearm - 2 * t_dikte_balk)**3)) / 12 # [m^4] # aanname dat de uitgaande arm een holle vierkante balk is, dus J = (b*h^3 - b_i*h_i^3)/12
print(f"traagheidsmoment uitgaande arm: {I_uitgaandearm:.6e} m^4")
# berekening van het dynamisch traagheidsmoment van de arm
J_uitgaandearm = (1/3) * (massa_uitgaandearm / (lengte_uitgaandearmx2 + lengte_uitgaandearmx1)) * (lengte_uitgaandearmx2**3 + lengte_uitgaandearmx1**3) # [kg*m^2] 
print(f"dynamisch traagheidsmoment uitgaande arm: {J_uitgaandearm:.2f} kg*m^2")
# berekening van het dynamisch traagheidsmoment van de massa last
J_massa_last = massa_last * (lengte_uitgaandearmx2)**2 # [kg*m^2]
print(f"traagheidsmoment massa last: {J_massa_last:.2f} kg*m^2")
# totale traagheidsmoment
J_totaal = J_uitgaandearm + J_massa_last # [kg*m^2]
print(f"totaal traagheidsmoment: {J_totaal:.2f} kg*m^2")

# opstellen momentvergelijkingen in verticale en horizontale richting
# horizontale richting: M = J * alpha
T_uitgaandeas = J_totaal * alpha_max # [N*m]
print(f"moment in horizontale richting: {T_uitgaandeas:.2f} N*m")
# verticale richting: M = m * g * r
g = 9.81 # [m/s^2]
M_uitgaandeas = massa_last * g * lengte_uitgaandearmx2 + massa_uitgaandearm * g * ((lengte_uitgaandearmx2 - lengte_uitgaandearmx1) / 2) # [N*m]
print(f"moment in verticale richting: {M_uitgaandeas:.2f} N*m")
# totale moment
M_totaal = ( T_uitgaandeas**2 + M_uitgaandeas**2 ) ** 0.5 # [N*m]
print(f"totaal moment: {M_totaal:.2f} N*m")
# berekening van maximale spanning in de uitgaande arm met behulp van weerstandsmoment
W_weerstand = I_uitgaandearm / (hoogte_uitgaandearm / 2) # [m^3] #hoogte of breedte
sigma_max = M_totaal / W_weerstand # [Pa]
print(f"maximale spanning in de uitgaande arm: {sigma_max:.2f} Pa")  

# berekening van de benodigde motorvermogen
P_motor = T_uitgaandeas * omega_max # [W]
print(f"benodigd motorvermogen: {P_motor:.2f} W")

alpha_max: 3.53 rad/s^2
omega_max: 2.36 rad/s
alpha_gem: 0.79 rad/s^2
omega_gem: 1.57 rad/s
massa uitgaande arm: 24.02 kg
traagheidsmoment uitgaande arm: 6.332500e-06 m^4
dynamisch traagheidsmoment uitgaande arm: 1.34 kg*m^2
traagheidsmoment massa last: 1.69 kg*m^2
totaal traagheidsmoment: 3.03 kg*m^2
moment in horizontale richting: 10.70 N*m
moment in verticale richting: 73.80 N*m
totaal moment: 74.58 N*m
maximale spanning in de uitgaande arm: 588837.05 Pa
benodigd motorvermogen: 25.22 W


## b. Uitgaande as

In [3]:
# gegevens
lengte_uitgaandeas1 = 0.069 # [m] # op basis van geometrische factoren, zonder lagerhoogte!
lengte_uitgaandeas2 = 0.11 # [m] # op basis van geometrische factoren
lengte_uitgaandeas3 = 0.124 # [m] # op basis van geometrische factoren, zonder lagerhoogte!
lengte_uitgaandeas = lengte_uitgaandeas1 + lengte_uitgaandeas2 + lengte_uitgaandeas3 # [m]
K_A_uitgaandeas = 1.5 # aanname, afhankelijk van de belasting en het ontwerp van de as, kan worden aangepast op basis van de specifieke omstandigheden
kerffactor_uitgaandeas = 2.3 # gemiddelde waarde voor persverbinding (met tandriemschijf) gebaseerd op R&M tabel 3-8 p54 tabellenboek
S_veiligheid_uitgaandeas = 2.0 
n_uitgaandeas = omega_max * 60 / (2 * np.pi) # [rpm] # omrekening van rad/s naar rpm
print(f"toerental van de uitgaande as: {n_uitgaandeas:.2f} rpm")
sigma_bD_uitgaandeas_Nmm2 = 400 / (kerffactor_uitgaandeas * S_veiligheid_uitgaandeas) # [N/mm^2] # voor 38Cr2
ro_staal = 7850 # [kg/m^3] # dichtheid van staal, herhaling

# snelheid en versnelling van de uitgaande as zijn gelijk
alpha_uitgaandeas_max = alpha_max # [rad/s^2]
omega_uitgaandeas_max = omega_max # [rad/s] 
alpha_uitgaandeas_gem = alpha_gem # [rad/s^2]
omega_uitgaandeas_gem = omega_gem # [rad/s]
print(f"maximale versnelling van de uitgaande as: {alpha_uitgaandeas_max:.2f} rad/s^2")
print(f"maximale snelheid van de uitgaande as: {omega_uitgaandeas_max:.2f} rad/s")
print(f"gemiddelde versnelling van de uitgaande as: {alpha_uitgaandeas_gem:.2f} rad/s^2")
print(f"gemiddelde snelheid van de uitgaande as: {omega_uitgaandeas_gem:.2f} rad/s")

# BEREKENING VOLGENS ROLOFF EN MATEK H11 -> MIN. ASDIAMETER
# STAP 1: Controle op sterkte (vermoeiing) via vergelijkmoment voor bepalen minimale asdiameter
# buigequivalent moment
M_beq_uitgaandeas = K_A_uitgaandeas * M_uitgaandeas # [N*m] 
print(f"equivalent buigmoment op de uitgaande as: {M_beq_uitgaandeas:.2f} N*m")
# torsie equivalent moment
T_eq_uitgaandeas = 9550 * ((P_motor/1000) / n_uitgaandeas) # [N*m] # omrekening van vermogen en toerental naar wringingsmoment
print(f"equivalent wringmoment op de uitgaande as: {T_eq_uitgaandeas:.2f} N*m")
# vergelijkmoment
M_vgl_uitgaandeas = ((M_beq_uitgaandeas)**2 + 0.75 * (0.7 * T_eq_uitgaandeas)**2)**(1/2)
print(f"vergelijkmoment op de uitgaande as: {M_vgl_uitgaandeas:.2f} N*m")
M_vgl_uitgaandeas_Nmm = M_vgl_uitgaandeas * (10**3) # [N*mm] # vergelijkmoment in N*mm voor vergelijking met de toelaatbare spanning in N/mm^2
print(f"vergelijkmoment op de uitgaande as: {M_vgl_uitgaandeas_Nmm:.2f} N*mm")
# voorlopige asdiameter
d_uitgaandeas_voorlopig = 3.4 * (M_vgl_uitgaandeas_Nmm / sigma_bD_uitgaandeas_Nmm2)**(1/3) # [mm] # voorlopige diameter van de uitgaande as op basis van het vergelijkmoment en de toelaatbare spanning, dit is een vereenvoudiging die kan worden aangepast op basis van de specifieke belasting en het ontwerp van de uitgaande as
print(f"voorlopige diameter van de uitgaande as volgens Roloff en Matek: {d_uitgaandeas_voorlopig:.2f} mm")
d_uitgaandeas_praktisch = 45 # [mm] # KIES!
print(f"praktische diameter van de uitgaande as: {d_uitgaandeas_praktisch:.2f} mm")
massa_uitgaandeas = ro_staal * (np.pi * (d_uitgaandeas_praktisch / 1000)**2 / 4) * lengte_uitgaandeas # [kg] # massa van de uitgaande as op basis van de dichtheid van staal, de voorlopige diameter en de lengte van de uitgaande as
print(f"massa van de uitgaande as: {massa_uitgaandeas:.2f} kg")

# berekening dynamisch massatraagheidsmoment J van de uitgaande as
J_uitgaandeas = (1/2) * massa_uitgaandeas * (d_uitgaandeas_praktisch / 1000)**2 # [kg*m^2] # aanname dat de uitgaande as een massieve cilinder is, dus J = (1/2) * m * r^2
print(f"dynamisch massatraagheidsmoment van de uitgaande as: {J_uitgaandeas:.2f} kg*m^2")


toerental van de uitgaande as: 22.50 rpm
maximale versnelling van de uitgaande as: 3.53 rad/s^2
maximale snelheid van de uitgaande as: 2.36 rad/s
gemiddelde versnelling van de uitgaande as: 0.79 rad/s^2
gemiddelde snelheid van de uitgaande as: 1.57 rad/s
equivalent buigmoment op de uitgaande as: 110.71 N*m
equivalent wringmoment op de uitgaande as: 10.70 N*m
vergelijkmoment op de uitgaande as: 110.90 N*m
vergelijkmoment op de uitgaande as: 110896.25 N*mm
voorlopige diameter van de uitgaande as volgens Roloff en Matek: 36.87 mm
praktische diameter van de uitgaande as: 45.00 mm
massa van de uitgaande as: 3.78 kg
dynamisch massatraagheidsmoment van de uitgaande as: 0.00 kg*m^2


## c. Tandriem
Berekeningen in deze sectie gebeuren volgens R&M H16 vb 16.4 p624

In [4]:
# gegevens voor de tandriem
K_A = 1.5 # aanname voor de veiligheidsfactor op de tandriem (start-stop bewegingen)
T_tandriem = T_uitgaandeas * K_A # [N*m] # aanname dat het wringingsmoment van de uitgaande as en de riem verwaarloosbaar is in vergelijking met degene van de uitgaande arm, dus het maximale moment op de tandriem is gelijk aan het maximale moment op de uitgaande as vermenigvuldigd met een veiligheidsfactor
print(f"maximaal moment op de tandriem: {T_tandriem:.2f} N*m")
alpha_tandriem_max = alpha_uitgaandeas_max
print(f"maximale versnelling van de tandriem: {alpha_tandriem_max:.2f} rad/s^2")
toerental_tandriem_max = (omega_uitgaandeas_max / (2 * np.pi)) * 60 # [rpm]
print(f"maximaal toerental van de tandriem: {toerental_tandriem_max:.2f} rpm")
# berekening vermogen van de tandriem, bedrijfsfactor K_A is al meegenomen in het berekenen van het maximale moment op de tandriem, dus we kunnen gewoon het maximale moment op de tandriem vermenigvuldigen met de maximale snelheid van de uitgaande as om het maximale vermogen van de tandriem te berekenen
P_tandriem = T_tandriem * omega_uitgaandeas_max # [W]
print(f"vermogen van de tandriem: {P_tandriem:.2f} W")
aantal_tandriemschijven = 2

# keuze tandriem
# VOORLOPIGE KEUZE (ZIE NOTA'S) PROFIEL T5 -> tabel 16.19
steek_tandriem = 5e-3 # [m] # steek van de tandriem, aanname voor profiel T5
aantal_tanden_tandriem = 150 # VRIJ TE KIEZEN, TUSSEN GRENZEN TABEL 16.19 aanname voor het aantal tanden op de tandriem, kan worden aangepast op basis van de berekening van het benodigde vermogen en de beschikbare tandriemen in de catalogus
tandhoogte_tandriem = 1.2e-3 # [m] # zie tabel
steekcirkeldiameter_tandriem = aantal_tanden_tandriem * steek_tandriem / (np.pi) # [m] #gelijk voor beide riemschijven aangezien de overbrengingsverhouding 1 is
print(f"steekcirkeldiameter van de tandriem: {steekcirkeldiameter_tandriem:.2f} m")
d_dk = (steek_tandriem / np.pi) * aantal_tanden_tandriem # [m] # diameter van de krukas van de tandriem, kan worden gebruikt om te controleren of de gekozen diameter van de uitgaande as geschikt is voor de tandriem
d_dg = d_dk # [m] # diameter van de generatoras van de tandriem, gelijk aan de diameter van de krukas aangezien de overbrengingsverhouding 1 is
print(f"diameter van de krukas (en andere diameter) van de tandriem is gelijk aan de steekcirkeldiameter")

# asafstand kiezen volgens geometrie en checken of deze binnen de voorwaarden R&M (16.21) p615 past
voorlopige_asafstand_tandriem = 0.3675 # [m] # aanname op basis van geometrische factoren
print(f"voorlopige asafstand van de tandriem: {voorlopige_asafstand_tandriem:.2f} m")
if (voorlopige_asafstand_tandriem) >= 0.5 * (d_dg + d_dk) + 0.015 and (voorlopige_asafstand_tandriem) <= 2 * (d_dg + d_dk):
    print("voorlopige asafstand binnen condities")
else:    print("voorlopige asafstand valt niet binnen condities")

# bepaling voorlopige riemlengte, daaruit volgt het rekenkundige aantal tanden
theoretische_riemlengte_tandriem = 2 * voorlopige_asafstand_tandriem + (np.pi * steekcirkeldiameter_tandriem) # [m] # theoretische lengte van de tandriem op basis van de asafstand en de steekcirkeldiameter, kan worden gebruikt om te controleren of de gekozen lengte van de tandriem geschikt is voor het ontwerp
print(f"theoretische lengte van de tandriem: {theoretische_riemlengte_tandriem:.2f} m")
rekenkundig_aantal_tanden_tandriem = (theoretische_riemlengte_tandriem / steek_tandriem) # [stuks] # rekenkundig aantal tanden op de tandriem op basis van de theoretische lengte en de steek, kan worden gebruikt om te controleren of het gekozen aantal tanden geschikt is voor het ontwerp
print(f"rekenkundig aantal tanden op de tandriem: {rekenkundig_aantal_tanden_tandriem:.2f} stuks")

# KIES AANTAL TANDEN VIA TABEL 16.19D, daaruit volgt de definitieve riemlengte
def_aantal_tanden_tandriem = 215 # [stuks] # definitief aantal tanden op de tandriem, gekozen op basis van de berekening van het benodigde vermogen en de beschikbare tandriemen in de catalogus
print(f"definitief aantal tanden op de tandriem: {def_aantal_tanden_tandriem} stuks")
def_lengte_tandriem = def_aantal_tanden_tandriem * steek_tandriem # [m] # definitieve lengte van de tandriem op basis van het definitieve aantal tanden en de steek, kan worden gebruikt om te controleren of de gekozen lengte van de tandriem geschikt is voor het ontwerp
print(f"definitieve lengte van de tandriem: {def_lengte_tandriem:.2f} m")

# bepaling definitieve asafstand
def_asafstand_tandriem = def_lengte_tandriem / 4 - ((np.pi / 8 ) * (d_dg + d_dk)) + ((def_lengte_tandriem / 4 - (np.pi / 8)*(d_dg + d_dk))**2 - (d_dg - d_dk)**2 / 8)**(1/2)
print(f"definitieve asafstand van de tandriem: {def_asafstand_tandriem:.2f} m")

# bepaling omspanningshoek (deze zijn gelijk voor beide schijven aangezien overbrengingsverhouding 1 is)
omspanningshoek_tandriem = np.pi # [rad] # want overbrengingsverhouding is 1, dus beide schijven hebben dezelfde diameter en de riem omsluit beide schijven volledig
print(f"omspanningshoek van de tandriem: {omspanningshoek_tandriem:.2f} rad")

# bepaling aantal ingrijpende tanden
# omzetten waarden van rad naar °
omspanningshoek_tandriem_graden = omspanningshoek_tandriem * (180 / np.pi) # [°] 
# berekening ingrijpende tanden met de waarden in °
tanden_in_ingreep = omspanningshoek_tandriem_graden / 360 * def_aantal_tanden_tandriem # [stuks] # aantal tanden van de tandriem die tegelijkertijd in ingreep zijn met de tandwielen, kan worden gebruikt om te controleren of er voldoende tanden in ingreep zijn om het benodigde vermogen over te brengen zonder slip
print(f"aantal tanden in ingreep: {tanden_in_ingreep:.2f} stuks")
# controle of er voldoende tanden in ingreep zijn, volgens de vuistregel moeten er minstens 6 tanden in ingreep zijn om slip te voorkomen, maar dit kan variëren afhankelijk van het benodigde vermogen en de belasting op de tandriem
if tanden_in_ingreep >= 6:
    print("er zijn voldoende tanden in ingreep om slip te voorkomen")
else:    print("er zijn niet voldoende tanden in ingreep om slip te voorkomen, overweeg een tandriem met meer tanden of een grotere diameter van de schijven")

# keuze riembreedte
riembreedte_tandriem = 16e-3 # [m] # voorlopig vrij gekozen uit tabel 16.19c # aanname voor de breedte van de tandriem, kan worden aangepast op basis van de berekening van het benodigde vermogen en de beschikbare tandriemen in de catalogus
print(f"breedte van de tandriem: {riembreedte_tandriem:.2f} m")
# BEKIJK OOK TOELAATBARE OMTREKSKRACHT TABEL 16.19C !!!

# bepaling snelheid en buigfrequentie
snelheid_tandriem = (steekcirkeldiameter_tandriem * np.pi * toerental_tandriem_max) / 60 # [m/s] # snelheid van de tandriem op basis van de steekcirkeldiameter en het toerental, kan worden gebruikt om te controleren of de gekozen tandriem geschikt is voor de benodigde snelheid
print(f"snelheid van de tandriem: {snelheid_tandriem:.2f} m/s")
buigfrequentie_tandriem = (snelheid_tandriem * aantal_tandriemschijven) / (def_lengte_tandriem) # [Hz] # buigfrequentie van de tandriem op basis van de snelheid, het aantal schijven en de lengte van de tandriem, kan worden gebruikt om te controleren of de gekozen tandriem geschikt is voor de benodigde buigfrequentie
print(f"buigfrequentie van de tandriem: {buigfrequentie_tandriem:.2f} Hz")

# massabepalingen
# massa tandriemschijven
massa_tandriemschijf = (np.pi * (steekcirkeldiameter_tandriem / 2)**2) * riembreedte_tandriem * ro_staal # [kg] # aanname dat de tandriemschijven gemaakt zijn van aluminium en een dikte hebben gelijk aan de dikte van de balken, kan worden gebruikt om te beoordelen of de belasting op de uitgaande as en de ondersteuningsplaten acceptabel is
print(f"massa van één tandriemschijf: {massa_tandriemschijf:.2f} kg")
totale_massa_tandriemschijven = massa_tandriemschijf * aantal_tandriemschijven # [kg] # totale massa van de tandriemschijven, kan worden gebruikt om te beoordelen of de belasting op de uitgaande as en de ondersteuningsplaten acceptabel is
print(f"totale massa van de tandriemschijven: {totale_massa_tandriemschijven:.2f} kg")

# askrachten
omtrekskracht_tandriem = (P_tandriem) / snelheid_tandriem # [N] # omtrekskracht van de tandriem op basis van het vermogen en de snelheid, kan worden gebruikt om te controleren of de gekozen tandriem geschikt is voor de benodigde omtrekskracht
print(f"omtrekskracht van de tandriem: {omtrekskracht_tandriem:.2f} N")
# om nu de askracht te berekenen gebruiken we tabel 16.19c en vergelijking 16.35
# opgelet, dit is de werkelijke waarde en niet de toegelaten waarde, toegelaten waarde zie tabel 16.19c
askracht_tandriem = 1.1 * omtrekskracht_tandriem # [N] # vgl 16.35 # aanname dat de askracht 10% hoger is dan de omtrekskracht, dit is een vuistregel die kan worden aangepast op basis van de specifieke belasting en het ontwerp van de tandriem
print(f"askracht van de tandriem: {askracht_tandriem:.2f} N")
# deze berekende askracht is dus de F_riem die eerder op de uitgaande as van toepassing was
# MANUELE CONTROLE NODIG VOOR MAX TOEGELATEN ASKRACHT VIA TABEL 16.19c
massa_tandriem_per_meter = 0.04 # [kg/m] # volgens specifiek gekozen component, zie datasheet
massa_tandriem = def_lengte_tandriem * massa_tandriem_per_meter # [kg] 
print(f"massa van de tandriem: {massa_tandriem:.2f} kg")

# berekening dynamisch massatraagheidsmoment J van de tandriemschijven en de tandriem
J_tandriemschijf = (1/2) * massa_tandriemschijf * (steekcirkeldiameter_tandriem / 2)**2 # [kg*m^2] # aanname dat de tandriemschijven massieve cilinders zijn, dus J = (1/2) * m * r^2
print(f"dynamisch massatraagheidsmoment van één tandriemschijf: {J_tandriemschijf:.2f} kg*m^2")
J_tandriem = massa_tandriem * (steekcirkeldiameter_tandriem / 2)**2 # [kg*m^2]
print(f"dynamisch massatraagheidsmoment van de tandriem: {J_tandriem:.2f} kg*m^2")

maximaal moment op de tandriem: 16.05 N*m
maximale versnelling van de tandriem: 3.53 rad/s^2
maximaal toerental van de tandriem: 22.50 rpm
vermogen van de tandriem: 37.83 W
steekcirkeldiameter van de tandriem: 0.24 m
diameter van de krukas (en andere diameter) van de tandriem is gelijk aan de steekcirkeldiameter
voorlopige asafstand van de tandriem: 0.37 m
voorlopige asafstand binnen condities
theoretische lengte van de tandriem: 1.48 m
rekenkundig aantal tanden op de tandriem: 297.00 stuks
definitief aantal tanden op de tandriem: 215 stuks
definitieve lengte van de tandriem: 1.07 m
definitieve asafstand van de tandriem: 0.16 m
omspanningshoek van de tandriem: 3.14 rad
aantal tanden in ingreep: 107.50 stuks
er zijn voldoende tanden in ingreep om slip te voorkomen
breedte van de tandriem: 0.02 m
snelheid van de tandriem: 0.28 m/s
buigfrequentie van de tandriem: 0.52 Hz
massa van één tandriemschijf: 5.62 kg
totale massa van de tandriemschijven: 11.24 kg
omtrekskracht van de tandriem: 134

## d. Spline-as

In [5]:
# gegevens spline-as uit massief staal
lengte_spline_as = 2.2 # [m]
diameter_spline_as = 0.045 # [m] # geitereerde waarde! Zie verder in dit blok code
K_A_spline_as = 1.5
lagerafstand = 0.380 # [m] # afstand tussen middelpunten van de groefkogellagers, op basis van geometrische factoren
sigma_bD = 245 # [N/mm^2] # volgens tabel 1-1 bij aanname van E295 staal
kerffactor_spline_as = 1.0 # geen kerf aanwezig in spline-as
S_veiligheid_spline_as = 2.0 
sigma_bD_spline_as = sigma_bD / (kerffactor_spline_as * S_veiligheid_spline_as) # [N/mm^2]

print("")
print("BUIGINGSANALYSE SPLINE-AS IN BEST CASE SCENARIO: RIEMKRACHT WORDT VOLLEDIG OPGEVANGEN DOOR ONDERSTEUNINGSPLATEN")
# BEREKENING VOLGENS ROLOFF EN MATEK H11
# buigequivalent moment
M_beq_spline = K_A_spline_as * askracht_tandriem * (lagerafstand / 4) # [N*m] # equivalent buigmoment op de spline-as op basis van de askracht, de veiligheidsfactor en de afstand tussen de lagers, dit is een vereenvoudiging die kan worden aangepast op basis van de specifieke belasting en het ontwerp van de spline-as
print(f"equivalent buigmoment op de spline-as: {M_beq_spline:.2f} N*m")
# vergelijkmoment
M_vgl_spline = ((M_beq_spline)**2 + 0.75 * (0.7 * T_tandriem)**2)**(1/2)
print(f"vergelijkmoment op de spline-as: {M_vgl_spline:.2f} N*m")
M_vgl_spline_Nmm = M_vgl_spline * (10**3) # [N*mm] # vergelijkmoment in N*mm voor vergelijking met de toelaatbare spanning in N/mm^2
print(f"vergelijkmoment op de spline-as: {M_vgl_spline_Nmm:.2f} N*mm")
# voorlopige asdiameter
d_spline_voorlopig = 3.4 * (M_vgl_spline_Nmm / sigma_bD_spline_as)**(1/3) # [mm] # voorlopige diameter van de spline-as op basis van het vergelijkmoment en de toelaatbare spanning, dit is een vereenvoudiging die kan worden aangepast op basis van de specifieke belasting en het ontwerp van de spline-as
print(f"voorlopige diameter van de spline-as volgens Roloff en Matek: {d_spline_voorlopig:.2f} mm")
d_spline_praktisch = 25 # [mm] # KIES!
print(f"praktische diameter van de spline-as: {d_spline_praktisch:.2f} mm")

print("")
# massaberekening spline-as
massa_spline = ro_staal * (np.pi * (d_spline_praktisch / 1000)**2 / 4) * lengte_spline_as # [kg] # massa van de spline-as op basis van de dichtheid van staal, de voorlopige diameter en de lengte van de spline-as
print(f"massa van de spline-as: {massa_spline:.2f} kg")

# berekening dynamisch massatraagheidsmoment J van de spline-as
J_spline_as = (1/2) * massa_spline * (d_spline_praktisch / 1000)**2 # [kg*m^2] # aanname van spline-as als massieve cilinder
print(f"dynamisch massatraagheidsmoment van de spline-as: {J_spline_as:.2f} kg*m^2")



BUIGINGSANALYSE SPLINE-AS IN BEST CASE SCENARIO: RIEMKRACHT WORDT VOLLEDIG OPGEVANGEN DOOR ONDERSTEUNINGSPLATEN
equivalent buigmoment op de spline-as: 21.08 N*m
vergelijkmoment op de spline-as: 23.22 N*m
vergelijkmoment op de spline-as: 23220.88 N*mm
voorlopige diameter van de spline-as volgens Roloff en Matek: 19.53 mm
praktische diameter van de spline-as: 25.00 mm

massa van de spline-as: 8.48 kg
dynamisch massatraagheidsmoment van de spline-as: 0.00 kg*m^2


## e. Ondersteuningsplaten en zijplaten + voor- en achterplaat

In [ ]:
# ONDERSTEUNINGSPLATEN
# RADIALE LASTEN: k_behuizing is een serieschakeling van de ondersteuningsplaten (aluminium), de groefkogellagers en de geleidende rails van M1
aantal_ondersteuningsplaten = 2
breedte_ondersteuningsplaat = 0.3 # [m] # In samenspraak met nota's # aanname 
lengte_ondersteuningsplaat = 0.7 # [m] # In samenspraak met nota's  # aanname
hoogte_ondersteuningsplaat = 0.03 # [m] # In samenspraak met nota's  # aanname
I_ondersteuningsplaat = (breedte_ondersteuningsplaat * hoogte_ondersteuningsplaat**3) / 12 # [m^4] # traagheidsmoment van de ondersteuningsplaat, aanname dat de ondersteuningsplaat een rechthoekig profiel heeft
E_ondersteuningsplaat = 70e9 # [Pa] # elasticiteitsmodulus van aluminium
overspanning_steunpunten = 0.072 # [m] # afstand tussen ophangingspunten centraal blokje in breedterichting, zie datasheet M1
# hieronder wordt het gat voor de kabelrups in de onderste ondersteuningsplaat mee in rekening gebracht -> belangrijk voor massabepaling
lengte_gat_ondersteplaat = 0.07 # [m]
breedte_gat_ondersteplaat = 0.1 # [m]
ro_aluminium = 2700 # [kg/m^3]

print("")
print("MASSABEPALING ONDERSTEUNINGSPLATEN")
# massa ondersteuningsplaten - vereenvoudigde berekening 
massa_ondersteuningsplaat_boven = breedte_ondersteuningsplaat * lengte_ondersteuningsplaat * hoogte_ondersteuningsplaat * ro_aluminium # [kg]
massa_ondersteuningsplaat_onder = (breedte_ondersteuningsplaat * lengte_ondersteuningsplaat * hoogte_ondersteuningsplaat - lengte_gat_ondersteplaat * breedte_gat_ondersteplaat * hoogte_ondersteuningsplaat) * ro_aluminium # [kg]
totale_massa_ondersteuningsplaten = massa_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder # [kg]
print(f"massa van de bovenste ondersteuningsplaat: {massa_ondersteuningsplaat_boven:.2f} kg")
print(f"massa van de onderste ondersteuningsplaat: {massa_ondersteuningsplaat_onder:.2f} kg")
print(f"totale massa van de ondersteuningsplaten: {totale_massa_ondersteuningsplaten:.2f} kg")

print("")
print("ZIJPLATEN")
# ZIJPLATEN
# nodige gegevens voor zijplaten
lagerhoogte = 0.031 # [m] # hoogte van lager deel 1
lagerafstand_uitgaandeas = lengte_uitgaandeas # [m]
breedte_kokerM1 = 0.2 # [m] # afhankelijk van ontwerp M1; bij gebrek aan een M1-koker is deze waarde de ideale waarde indien enkel de geometrie van M2 in beschouwing genomen zou worden
speling_zijplaten = 0.015 # [m] # aanname speling
aantal_zijplaten = 2

# zijplaatjes - te verdelen in twee delen: A/ zijplaat bij deel M1 B/ zijplaat M2 zichtbaar
# bepaling afmetingen
hoogte_zijplaatA = lagerafstand_uitgaandeas + 2 * lagerhoogte + 2 * hoogte_ondersteuningsplaat # [m] 
hoogte_zijplaatB = hoogte_zijplaatA
hoogte_inkeping_zijplaatB = hoogte_uitgaandearm + 2 * speling_zijplaten # [m]
dikte_zijplaatA = 0.001 # [m] # aanname
dikte_zijplaatB = 0.001 # [m] # aanname
lengte_zijplaatA = breedte_kokerM1 + speling_zijplaten
lengte_zijplaatB = lengte_ondersteuningsplaat - lengte_zijplaatA
lengte_inkeping_zijplaatB = lengte_uitgaandearmx1 + 0.1 + speling_zijplaten # [m] # de 0.1 is een aanname voor extra speling
oppervlakte_zijplaatA = hoogte_zijplaatA * lengte_zijplaatA # [m^2]
volume_zijplaatA = oppervlakte_zijplaatA * dikte_zijplaatA # [m^3]
oppervlakte_zijplaatB = (hoogte_zijplaatB * lengte_zijplaatB) - (hoogte_inkeping_zijplaatB * lengte_inkeping_zijplaatB) # [m^2]
volume_zijplaatB = oppervlakte_zijplaatB * dikte_zijplaatB # [m^3]

# bepaling massa zijplaten
massa_zijplaatA = volume_zijplaatA * ro_aluminium # [kg]
massa_zijplaatB = volume_zijplaatB * ro_aluminium # [kg]
totale_massa_zijplaten = aantal_zijplaten * (massa_zijplaatA + massa_zijplaatB) # [kg] # totale massa van de zijplaten
print(f"massa van één zijplaat A: {massa_zijplaatA:.2f} kg")
print(f"massa van één zijplaat B: {massa_zijplaatB:.2f} kg")
print(f"totale massa van de zijplaten: {totale_massa_zijplaten:.2f} kg")

# voorplaat - berekend als de sectie boven en onder de arm samen
hoogte_voorplaat = hoogte_zijplaatB - hoogte_inkeping_zijplaatB # [m]
dikte_voorplaat = dikte_zijplaatB # [m]
lengte_voorplaat = breedte_ondersteuningsplaat # [m]
oppervlakte_voorplaat = hoogte_voorplaat * lengte_voorplaat # [m^2]
volume_voorplaat = oppervlakte_voorplaat * dikte_voorplaat # [m^3]
massa_voorplaat = volume_voorplaat * ro_aluminium # [kg]
print(f"massa van de voorplaat: {massa_voorplaat:.2f} kg")

# achterplaat - berekend worst case scenario zonder gaten
hoogte_achterplaat = hoogte_zijplaatB # [m]
dikte_achterplaat = 0.02 # [m] # bepaald, zie verder
lengte_achterplaat = breedte_ondersteuningsplaat # [m]
oppervlakte_achterplaat = hoogte_achterplaat * lengte_achterplaat # [m^2]
volume_achterplaat = oppervlakte_achterplaat * dikte_achterplaat # [m^3]
massa_achterplaat = volume_achterplaat * ro_staal # [kg]
print(f"massa van de achterplaat: {massa_achterplaat:.2f} kg")


MASSABEPALING ONDERSTEUNINGSPLATEN
massa van de bovenste ondersteuningsplaat: 17.01 kg
massa van de onderste ondersteuningsplaat: 16.44 kg
totale massa van de ondersteuningsplaten: 33.45 kg

ZIJPLATEN
massa van één zijplaat A: 0.25 kg
massa van één zijplaat B: 0.47 kg
totale massa van de zijplaten: 1.43 kg
massa van de voorplaat: 0.24 kg
massa van de achterplaat: 6.89 kg


## f. Kleinere componenten
Overzicht van kleinere componenten voor massaberekening

In [ ]:
# axiale lagers - catalogusgedeelte
massa_axiale_lager = 1.3 # [kg] # volgens datasheet

# axiale lagers - zelf ontworpen gedeelte
lengte_rechthoekigdeel_axlager = 0.140 # [m]
breedte_rechthoekigdeel_axlager = 0.11 # [m]
hoogte_rechthoekigdeel_axlager = 0.016 # [m]
binnendiameter_cilindrischdeel_axlager = 0.045 # [m]
hoogte_cilindrischdeel_axlager = 0.020 # [m]
dikte_cilindrischdeel_axlager = 0.0075 # [m]
buitendiameter_cilindrischdeel_axlager = binnendiameter_cilindrischdeel_axlager + 2 * dikte_cilindrischdeel_axlager # [m]
volume_axlager = (lengte_rechthoekigdeel_axlager * breedte_rechthoekigdeel_axlager * hoogte_rechthoekigdeel_axlager) + (np.pi * (buitendiameter_cilindrischdeel_axlager / 2)**2 * hoogte_cilindrischdeel_axlager) - (np.pi * (binnendiameter_cilindrischdeel_axlager / 2)**2 * (hoogte_cilindrischdeel_axlager + hoogte_rechthoekigdeel_axlager)) # [m^3]
massa_axlager_zelf = volume_axlager * ro_staal # [kg]

aantal_axiale_lagers = 3
totale_massa_axiale_lagers = (massa_axiale_lager + massa_axlager_zelf) * aantal_axiale_lagers # [kg]

# hoeksensor
massa_hoeksensor = 0.1 # [kg] # volgens datasheet

# spline(flens)moeren
massa_splineflensmoer = 0.34 # [kg] # volgens datasheet # OPGELET geldt voor standaard component van 34mm, voor dit ontwerp is bewerking tot 35mm nodig
aantal_splineflensmoeren = 2
totale_massa_splineflensmoeren = massa_splineflensmoer * aantal_splineflensmoeren # [kg]

massa_splinehub = 0.36 # [kg] # volgens datasheet
aantal_splinehubs = 7
totale_massa_splinehubs = massa_splinehub * aantal_splinehubs # [kg]

# groefkogellagers
massa_groefkogellager = 0.305 # [kg] # volgens datasheet
aantal_groefkogellagers = 2
totale_massa_groefkogellagers = massa_groefkogellager * aantal_groefkogellagers # [kg]

# balgkoppelingen
massa_balgkoppeling = 0.8185 # [kg] # volgens datasheet
aantal_balgkoppelingen = 2
totale_massa_balgkoppelingen = massa_balgkoppeling * aantal_balgkoppelingen # [kg]

# overgangscilindertje bovenaan spline-as
diameter_boven_ovcilinderboven = 0.038 # [m]
lengte_boven_ovcilinderboven = 0.0696 # [m]
diameter_onder_ovcilinderboven = 0.032 # [m]
lengte_onder_ovcilinderboven = 0.0624 # [m]
diameter_gat_ovcilinderboven = 0.020 # [m] 
lengte_gat_ovcilinderboven = 0.040 # [m]
# benaderend, spiegat niet meegerekend
volume_ovboven = (np.pi * (diameter_boven_ovcilinderboven / 2)**2) * lengte_boven_ovcilinderboven + (np.pi * (diameter_onder_ovcilinderboven / 2)**2) * lengte_onder_ovcilinderboven - (np.pi * (diameter_gat_ovcilinderboven / 2)**2) * lengte_gat_ovcilinderboven # [m^3]
massa_ovcilinderboven = volume_ovboven * ro_staal # [kg]

# overgangscilindertje onderaan spline-as
diameter_boven_ovcilinderonder = 0.032 # [m]
lengte_boven_ovcilinderonder = 0.070 # [m]
diameter_onder_ovcilinderonder = 0.045 # [m]
lengte_onder_ovcilinderonder = 0.086 # [m]
volume_ovonder = (np.pi * (diameter_boven_ovcilinderonder / 2)**2) * lengte_boven_ovcilinderonder + (np.pi * (diameter_onder_ovcilinderonder / 2)**2) * lengte_onder_ovcilinderonder # [m^3]
massa_ovcilinderonder = volume_ovonder * ro_staal # [kg]

## g. Totaalberekeningen


In [8]:
# totaal dynamisch traagheidsmoment J van het systeem -> bepaling totaal motorvermogen
print("")
print("MOTORSELECTIE")
J_totaal = J_massa_last + J_uitgaandearm + J_uitgaandeas + J_tandriemschijf * aantal_tandriemschijven + J_tandriem + J_spline_as # [kg*m^2]
print(f"totale dynamische traagheidsmoment van het systeem: {J_totaal:.2f} kg*m^2")
T_totaal = J_totaal * alpha_max # [N*m]
print(f"totaal motorvermogen op basis van het totale dynamische traagheidsmoment: {T_totaal:.2f} N*m")
P_motor = T_totaal * omega_max # [W]
print(f"totaal motorvermogen op basis van het totale dynamische traagheidsmoment: {P_motor:.2f} W")
massa_motor = 15.51 # [kg] # zie datasheet motor

# totale massa van het systeem
print("")
print("TOTALE MASSA")
massa_totaal = massa_uitgaandearm + massa_uitgaandeas + totale_massa_axiale_lagers + massa_hoeksensor + totale_massa_tandriemschijven + massa_tandriem + massa_spline + totale_massa_splinehubs + totale_massa_splineflensmoeren + totale_massa_groefkogellagers + totale_massa_balgkoppelingen + massa_ovcilinderboven + massa_ovcilinderonder + massa_motor + totale_massa_ondersteuningsplaten + totale_massa_zijplaten + massa_achterplaat + massa_voorplaat  # [kg] # totale massa van het systeem
print(f"totale massa van het systeem: {massa_totaal:.2f} kg")
massa_buigmomentM1 = massa_totaal - massa_spline - (massa_axiale_lager + massa_axlager_zelf) - 2 * massa_splinehub - totale_massa_balgkoppelingen - massa_ovcilinderboven - massa_ovcilinderonder - massa_motor # [kg] # massa die zal inwerken op het montageplaatje van M1
print(f"totale massa van het systeem zonder de spline-as: {massa_buigmomentM1:.2f} kg")

# invloed op M1
# oorsprong coördinatenstelsel is het centrale plaatje op M1, die in het midden volgens de y-as in de constructie van M2 gesitueerd is
# ZWAARTEPUNT VIA COÖRDINATENBEREKENING
print("")
print("BEPALING ZWAARTEPUNT ZONDER SPLINE-AS")
# voor sit1 gedefinieerde coördinaten op 0°
coördinaat_x_uitgaandearm_sit1 = 0.69 # [m]
coördinaat_y_uitgaandearm_sit1 = 0.0 # [m]
coördinaat_z_uitgaandearm_sit1 = -0.0275 # [m]
# voor sit2 gedefinieerde coördinaten op +90°
coördinaat_x_uitgaandearm_sit2 = 0.53 # [m]
coördinaat_y_uitgaandearm_sit2 = 0.13 # [m]
coördinaat_z_uitgaandearm_sit2 = -0.0275 # [m]
# voor sit3 gedefinieerde coördinaten op -90°
coördinaat_x_uitgaandearm_sit3 = 0.53 # [m]
coördinaat_y_uitgaandearm_sit3 = -0.13 # [m]
coördinaat_z_uitgaandearm_sit3 = -0.0275 # [m]

coördinaat_x_uitgaandeas = 0.53 # [m]
coördinaat_y_uitgaandeas = 0.0 # [m]
coördinaat_z_uitgaandeas = 0.0 # [m]

coördinaat_x_axialelagerboven = 0.53 # [m]
coördinaat_y_axialelagerboven = 0.0 # [m]
coördinaat_z_axialelagerboven = 0.1525 # [m]

coördinaat_x_axialelageronder = 0.53 # [m]
coördinaat_y_axialelageronder = 0.0 # [m]
coördinaat_z_axialelageronder = -0.1525 # [m]

coördinaat_x_hoeksensor = 0.53 # [m]
coördinaat_y_hoeksensor = 0.0 # [m]
coördinaat_z_hoeksensor = -0.1025 # [m]

coördinaat_x_tandriemschijf2 = 0.53 # [m]
coördinaat_y_tandriemschijf2 = 0.0 # [m]
coördinaat_z_tandriemschijf2 = 0.0825 # [m]

coördinaat_x_tandriemschijf1 = 0.1625 # [m]
coördinaat_y_tandriemschijf1 = 0.0 # [m]
coördinaat_z_tandriemschijf1 = 0.0825 # [m]

coördinaat_x_ondersteuningsplaat_boven = 0.35 # [m]
coördinaat_y_ondersteuningsplaat_boven = 0.0 # [m]
coördinaat_z_ondersteuningsplaat_boven = 0.1975 # [m]

coördinaat_x_ondersteuningsplaat_beneden = 0.35 # [m]
coördinaat_y_ondersteuningsplaat_beneden = 0.0 # [m]
coördinaat_z_ondersteuningsplaat_beneden = -0.1975 # [m]

coördinaat_x_splineflensmoer1 = 0.1625 # [m]
coördinaat_y_splineflensmoer1 = 0.0 # [m]
coördinaat_z_splineflensmoer1 = 0.1175 # [m]

coördinaat_x_splineflensmoer2 = 0.1625 # [m]
coördinaat_y_splineflensmoer2 = 0.0 # [m]
coördinaat_z_splineflensmoer2 = 0.0475 # [m]

coördinaat_x_splinedhub1 = 0.1625 # [m]
coördinaat_y_splinedhub1 = 0.0 # [m]
coördinaat_z_splinedhub1 = 0.1625 # [m]

coördinaat_x_splinedhub2 = 0.1625 # [m]
coördinaat_y_splinedhub2 = 0.0 # [m]
coördinaat_z_splinedhub2 = -0.005 # [m]

coördinaat_x_splinedhub3 = 0.1625 # [m]
coördinaat_y_splinedhub3 = 0.0 # [m]
coördinaat_z_splinedhub3 = -0.06 # [m]

coördinaat_x_splinedhub4 = 0.1625 # [m]
coördinaat_y_splinedhub4 = 0.0 # [m]
coördinaat_z_splinedhub4 = -0.115 # [m]

coördinaat_x_splinedhub5 = 0.1625 # [m]
coördinaat_y_splinedhub5 = 0.0 # [m]
coördinaat_z_splinedhub5 = -0.17 # [m]

coördinaat_x_achterplaat = -0.01 # [m]
coördinaat_y_achterplaat = 0.0 # [m]
coördinaat_z_achterplaat = 0.0 # [m]

coördinaat_x_zijplaten = 0.35 # [m]
coördinaat_y_zijplaten = 0.0 # [m]
coördinaat_z_zijplaten = 0.0 # [m]

coördinaat_x_voorplaat = 0.7005 # [m]
coördinaat_y_voorplaat = 0.0 # [m]
coördinaat_z_voorplaat = 0.01625 # [m]

print("SITUATIE 1: ZWAARTEPUNT ZONDER SPLINE-AS EN MOMENTEN OP M1 BIJ GEALIGNEERDE UITGAANDE ARM (0°)")
zwaartepunt_x_sit1 = (massa_uitgaandearm * coördinaat_x_uitgaandearm_sit1 + massa_uitgaandeas * coördinaat_x_uitgaandeas + massa_tandriemschijf * coördinaat_x_tandriemschijf2 + massa_tandriemschijf * coördinaat_x_tandriemschijf1 + massa_ondersteuningsplaat_boven * coördinaat_x_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder * coördinaat_x_ondersteuningsplaat_beneden + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_x_axialelagerboven + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_x_axialelageronder + massa_hoeksensor * coördinaat_x_hoeksensor + massa_splineflensmoer * coördinaat_x_splineflensmoer1 + massa_splineflensmoer * coördinaat_x_splineflensmoer2 + massa_splinehub * coördinaat_x_splinedhub1 + massa_splinehub * coördinaat_x_splinedhub2 + massa_splinehub * coördinaat_x_splinedhub3 + massa_splinehub * coördinaat_x_splinedhub4 + massa_splinehub * coördinaat_x_splinedhub5 + massa_achterplaat * coördinaat_x_achterplaat + totale_massa_zijplaten * coördinaat_x_zijplaten + massa_voorplaat * coördinaat_x_voorplaat) / massa_buigmomentM1 # [m]
zwaartepunt_y_sit1 = (massa_uitgaandearm * coördinaat_y_uitgaandearm_sit1 + massa_uitgaandeas * coördinaat_y_uitgaandeas + massa_tandriemschijf * coördinaat_y_tandriemschijf2 + massa_tandriemschijf * coördinaat_y_tandriemschijf1 + massa_ondersteuningsplaat_boven * coördinaat_y_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder * coördinaat_y_ondersteuningsplaat_beneden + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_y_axialelagerboven + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_y_axialelageronder + massa_hoeksensor * coördinaat_y_hoeksensor + massa_splineflensmoer * coördinaat_y_splineflensmoer1 + massa_splineflensmoer * coördinaat_y_splineflensmoer2 + massa_splinehub * coördinaat_y_splinedhub1 + massa_splinehub * coördinaat_y_splinedhub2 + massa_splinehub * coördinaat_y_splinedhub3 + massa_splinehub * coördinaat_y_splinedhub4 + massa_splinehub * coördinaat_y_splinedhub5 + massa_achterplaat * coördinaat_y_achterplaat + totale_massa_zijplaten * coördinaat_y_zijplaten + massa_voorplaat * coördinaat_y_voorplaat) / massa_buigmomentM1 # [m]
zwaartepunt_z_sit1 = (massa_uitgaandearm * coördinaat_z_uitgaandearm_sit1 + massa_uitgaandeas * coördinaat_z_uitgaandeas + massa_tandriemschijf * coördinaat_z_tandriemschijf2 + massa_tandriemschijf * coördinaat_z_tandriemschijf1 + massa_ondersteuningsplaat_boven * coördinaat_z_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder * coördinaat_z_ondersteuningsplaat_beneden + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_z_axialelagerboven + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_z_axialelageronder + massa_hoeksensor * coördinaat_z_hoeksensor + massa_splineflensmoer * coördinaat_z_splineflensmoer1 + massa_splineflensmoer * coördinaat_z_splineflensmoer2 + massa_splinehub * coördinaat_z_splinedhub1 + massa_splinehub * coördinaat_z_splinedhub2 + massa_splinehub * coördinaat_z_splinedhub3 + massa_splinehub * coördinaat_z_splinedhub4 + massa_splinehub * coördinaat_z_splinedhub5 + massa_achterplaat * coördinaat_z_achterplaat + totale_massa_zijplaten * coördinaat_z_zijplaten + massa_voorplaat * coördinaat_z_voorplaat) / massa_buigmomentM1 # [m]
print(f"zwaartepunt van het systeem in situatie 1 (zonder spline-as; uitgaande arm gealigneerd): ({zwaartepunt_x_sit1:.2f}, {zwaartepunt_y_sit1:.2f}, {zwaartepunt_z_sit1:.2f}) m")

# situatie 1 is wanneer de uitgaande arm gealigneerd staat met de ondersteuningsplaten (0°)
moment_M1_x_sit1 = massa_buigmomentM1 * 9.81 * zwaartepunt_y_sit1 # [N*m]
moment_M1_y_sit1 = massa_buigmomentM1 * 9.81 * zwaartepunt_x_sit1 # [N*m]
moment_M1_z_sit1 = 0 # [N*m]
print(f"momenten op M1 in situatie 1 (zonder spline-as; uitgaande arm gealigneerd): Mx = {moment_M1_x_sit1:.2f} N*m, My = {moment_M1_y_sit1:.2f} N*m, Mz = {moment_M1_z_sit1:.2f} N*m")

print("")
print("SITUATIE 2: ZWAARTEPUNT ZONDER SPLINE-AS EN MOMENTEN OP M1 BIJ LODRECHTE UITGAANDE ARM (90°)")
# situatie 2 is wanneer de uitgaande arm loodrecht op de lijn van de ondersteuningsplaten staat (90° of -90°)
zwaartepunt_x_sit2 = (massa_uitgaandearm * coördinaat_x_uitgaandearm_sit2 + massa_uitgaandeas * coördinaat_x_uitgaandeas + massa_tandriemschijf * coördinaat_x_tandriemschijf2 + massa_tandriemschijf * coördinaat_x_tandriemschijf1 + massa_ondersteuningsplaat_boven * coördinaat_x_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder * coördinaat_x_ondersteuningsplaat_beneden + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_x_axialelagerboven + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_x_axialelageronder + massa_hoeksensor * coördinaat_x_hoeksensor + massa_splineflensmoer * coördinaat_x_splineflensmoer1 + massa_splineflensmoer * coördinaat_x_splineflensmoer2 + massa_splinehub * coördinaat_x_splinedhub1 + massa_splinehub * coördinaat_x_splinedhub2 + massa_splinehub * coördinaat_x_splinedhub3 + massa_splinehub * coördinaat_x_splinedhub4 + massa_splinehub * coördinaat_x_splinedhub5 + massa_achterplaat * coördinaat_x_achterplaat + totale_massa_zijplaten * coördinaat_x_zijplaten + massa_voorplaat * coördinaat_x_voorplaat) / massa_buigmomentM1 # [m]
zwaartepunt_y_sit2 = (massa_uitgaandearm * coördinaat_y_uitgaandearm_sit2 + massa_uitgaandeas * coördinaat_y_uitgaandeas + massa_tandriemschijf * coördinaat_y_tandriemschijf2 + massa_tandriemschijf * coördinaat_y_tandriemschijf1 + massa_ondersteuningsplaat_boven * coördinaat_y_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder * coördinaat_y_ondersteuningsplaat_beneden + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_y_axialelagerboven + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_y_axialelageronder + massa_hoeksensor * coördinaat_y_hoeksensor + massa_splineflensmoer * coördinaat_y_splineflensmoer1 + massa_splineflensmoer * coördinaat_y_splineflensmoer2 + massa_splinehub * coördinaat_y_splinedhub1 + massa_splinehub * coördinaat_y_splinedhub2 + massa_splinehub * coördinaat_y_splinedhub3 + massa_splinehub * coördinaat_y_splinedhub4 + massa_splinehub * coördinaat_y_splinedhub5 + massa_achterplaat * coördinaat_y_achterplaat + totale_massa_zijplaten * coördinaat_y_zijplaten + massa_voorplaat * coördinaat_y_voorplaat) / massa_buigmomentM1 # [m]
zwaartepunt_z_sit2 = (massa_uitgaandearm * coördinaat_z_uitgaandearm_sit2 + massa_uitgaandeas * coördinaat_z_uitgaandeas + massa_tandriemschijf * coördinaat_z_tandriemschijf2 + massa_tandriemschijf * coördinaat_z_tandriemschijf1 + massa_ondersteuningsplaat_boven * coördinaat_z_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder * coördinaat_z_ondersteuningsplaat_beneden + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_z_axialelagerboven + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_z_axialelageronder + massa_hoeksensor * coördinaat_z_hoeksensor + massa_splineflensmoer * coördinaat_z_splineflensmoer1 + massa_splineflensmoer * coördinaat_z_splineflensmoer2 + massa_splinehub * coördinaat_z_splinedhub1 + massa_splinehub * coördinaat_z_splinedhub2 + massa_splinehub * coördinaat_z_splinedhub3 + massa_splinehub * coördinaat_z_splinedhub4 + massa_splinehub * coördinaat_z_splinedhub5 + massa_achterplaat * coördinaat_z_achterplaat + totale_massa_zijplaten * coördinaat_z_zijplaten + massa_voorplaat * coördinaat_z_voorplaat) / massa_buigmomentM1 # [m]
print(f"zwaartepunt van het systeem in situatie 2 (zonder spline-as; uitgaande arm loodrecht): ({zwaartepunt_x_sit2:.2f}, {zwaartepunt_y_sit2:.2f}, {zwaartepunt_z_sit2:.2f}) m")

moment_M1_x_sit2 = massa_buigmomentM1 * 9.81 * zwaartepunt_y_sit2 # [N*m]
moment_M1_y_sit2 = massa_buigmomentM1 * 9.81 * zwaartepunt_x_sit2 # [N*m]
moment_M1_z_sit2 = 0 # [N*m]
print(f"momenten op M1 in situatie 2 (zonder spline-as; uitgaande arm loodrecht): Mx = {moment_M1_x_sit2:.2f} N*m, My = {moment_M1_y_sit2:.2f} N*m, Mz = {moment_M1_z_sit2:.2f} N*m")

print("")
print("SITUATIE 3: ZWAARTEPUNT ZONDER SPLINE-AS EN MOMENTEN OP M1 BIJ TEGENOVERGESTELDE LODRECHTE UITGAANDE ARM (-90°)")
# situatie 3 is wanneer de uitgaande arm loodrecht op de lijn van de ondersteuningsplaten staat in de tegenovergestelde richting van situatie 2 (-90°)
zwaartepunt_x_sit3 = (massa_uitgaandearm * coördinaat_x_uitgaandearm_sit3 + massa_uitgaandeas * coördinaat_x_uitgaandeas + massa_tandriemschijf * coördinaat_x_tandriemschijf2 + massa_tandriemschijf * coördinaat_x_tandriemschijf1 + massa_ondersteuningsplaat_boven * coördinaat_x_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder * coördinaat_x_ondersteuningsplaat_beneden + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_x_axialelagerboven + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_x_axialelageronder + massa_hoeksensor * coördinaat_x_hoeksensor + massa_splineflensmoer * coördinaat_x_splineflensmoer1 + massa_splineflensmoer * coördinaat_x_splineflensmoer2 + massa_splinehub * coördinaat_x_splinedhub1 + massa_splinehub * coördinaat_x_splinedhub2 + massa_splinehub * coördinaat_x_splinedhub3 + massa_splinehub * coördinaat_x_splinedhub4 + massa_splinehub * coördinaat_x_splinedhub5 + massa_achterplaat * coördinaat_x_achterplaat + totale_massa_zijplaten * coördinaat_x_zijplaten + massa_voorplaat * coördinaat_x_voorplaat) / massa_buigmomentM1 # [m]
zwaartepunt_y_sit3 = (massa_uitgaandearm * coördinaat_y_uitgaandearm_sit3 + massa_uitgaandeas * coördinaat_y_uitgaandeas + massa_tandriemschijf * coördinaat_y_tandriemschijf2 + massa_tandriemschijf * coördinaat_y_tandriemschijf1 + massa_ondersteuningsplaat_boven * coördinaat_y_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder * coördinaat_y_ondersteuningsplaat_beneden + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_y_axialelagerboven + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_y_axialelageronder + massa_hoeksensor * coördinaat_y_hoeksensor + massa_splineflensmoer * coördinaat_y_splineflensmoer1 + massa_splineflensmoer * coördinaat_y_splineflensmoer2 + massa_splinehub * coördinaat_y_splinedhub1 + massa_splinehub * coördinaat_y_splinedhub2 + massa_splinehub * coördinaat_y_splinedhub3 + massa_splinehub * coördinaat_y_splinedhub4 + massa_splinehub * coördinaat_y_splinedhub5 + massa_achterplaat * coördinaat_y_achterplaat + totale_massa_zijplaten * coördinaat_y_zijplaten + massa_voorplaat * coördinaat_y_voorplaat) / massa_buigmomentM1 # [m]
zwaartepunt_z_sit3 = (massa_uitgaandearm * coördinaat_z_uitgaandearm_sit3 + massa_uitgaandeas * coördinaat_z_uitgaandeas + massa_tandriemschijf * coördinaat_z_tandriemschijf2 + massa_tandriemschijf * coördinaat_z_tandriemschijf1 + massa_ondersteuningsplaat_boven * coördinaat_z_ondersteuningsplaat_boven + massa_ondersteuningsplaat_onder * coördinaat_z_ondersteuningsplaat_beneden + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_z_axialelagerboven + (massa_axiale_lager + massa_axlager_zelf) * coördinaat_z_axialelageronder + massa_hoeksensor * coördinaat_z_hoeksensor + massa_splineflensmoer * coördinaat_z_splineflensmoer1 + massa_splineflensmoer * coördinaat_z_splineflensmoer2 + massa_splinehub * coördinaat_z_splinedhub1 + massa_splinehub * coördinaat_z_splinedhub2 + massa_splinehub * coördinaat_z_splinedhub3 + massa_splinehub * coördinaat_z_splinedhub4 + massa_splinehub * coördinaat_z_splinedhub5 + massa_achterplaat * coördinaat_z_achterplaat + totale_massa_zijplaten * coördinaat_z_zijplaten + massa_voorplaat * coördinaat_z_voorplaat) / massa_buigmomentM1 # [m]
print(f"zwaartepunt van het systeem in situatie 3 (zonder spline-as; uitgaande arm tegenovergesteld loodrecht): ({zwaartepunt_x_sit3:.2f}, {zwaartepunt_y_sit3:.2f}, {zwaartepunt_z_sit3:.2f}) m")

moment_M1_x_sit3 = massa_buigmomentM1 * 9.81 * zwaartepunt_y_sit3 # [N*m]
moment_M1_y_sit3 = massa_buigmomentM1 * 9.81 * zwaartepunt_x_sit3 # [N*m]
moment_M1_z_sit3 = 0 # [N*m]
print(f"momenten op M1 in situatie 3 (zonder spline-as; uitgaande arm tegenovergesteld loodrecht): Mx = {moment_M1_x_sit3:.2f} N*m, My = {moment_M1_y_sit3:.2f} N*m, Mz = {moment_M1_z_sit3:.2f} N*m")


MOTORSELECTIE
totale dynamische traagheidsmoment van het systeem: 3.12 kg*m^2
totaal motorvermogen op basis van het totale dynamische traagheidsmoment: 11.01 N*m
totaal motorvermogen op basis van het totale dynamische traagheidsmoment: 25.94 W

TOTALE MASSA
totale massa van het systeem: 122.38 kg
totale massa van het systeem zonder de spline-as: 90.80 kg

BEPALING ZWAARTEPUNT ZONDER SPLINE-AS
SITUATIE 1: ZWAARTEPUNT ZONDER SPLINE-AS EN MOMENTEN OP M1 BIJ GEALIGNEERDE UITGAANDE ARM (0°)
zwaartepunt van het systeem in situatie 1 (zonder spline-as; uitgaande arm gealigneerd): (0.43, 0.00, 0.00) m
momenten op M1 in situatie 1 (zonder spline-as; uitgaande arm gealigneerd): Mx = 0.00 N*m, My = 379.54 N*m, Mz = 0.00 N*m

SITUATIE 2: ZWAARTEPUNT ZONDER SPLINE-AS EN MOMENTEN OP M1 BIJ LODRECHTE UITGAANDE ARM (90°)
zwaartepunt van het systeem in situatie 2 (zonder spline-as; uitgaande arm loodrecht): (0.38, 0.03, 0.00) m
momenten op M1 in situatie 2 (zonder spline-as; uitgaande arm loodrecht): 

## h. Perspassing tandriemschijf 2 - uitgaande as

In [9]:
# gegevens
n_as_pers1 = n_uitgaandeas # [rpm]
P_as_pers1 = T_uitgaandeas * (n_as_pers1 * 2 * np.pi / 60) # [W]
d_as_pers1 = 0.045 # [m]

# bepaling interval naaflengte (tandriemschijf 2)
naaflengte_interval_min_pers1 = 0.8* d_as_pers1 # [m] # voor staal
naaflengte_interval_max_pers1 = 1.0* d_as_pers1 # [m] # voor staal
print("PERSPASSING 1: TANDRIEMSCHIJF 2 - UITGAANDE AS")
print("")
print("BEPALING INTERVAL NAAFLENGTE (TANDRIEMSCHIJF 2)")
print(f"interval naaflengte voor tandriemschijf 2: [{naaflengte_interval_min_pers1:.4f}, {naaflengte_interval_max_pers1:.4f}] m")
naaflengte_praktisch_pers1 = naaflengte_interval_min_pers1 # [m] # dit is dus de breedte van de tandriemschijf op de uitgaande as
print(f"praktische naaflengte voor tandriemschijf 2: {naaflengte_praktisch_pers1:.4f} m")

# contactoppervlakte
oppervlakte_contact_pers1 = naaflengte_praktisch_pers1 * np.pi * d_as_pers1 # [m^2]
print("")
print("CONTACTOPPERVLAKTE PERSPASSING 1")
print(f"contactoppervlakte tussen tandriemschijf 2 en uitgaande as: {oppervlakte_contact_pers1:.4f} m^2")

# totale inwerkende kracht op de as - bekijk radiale en tangentiële krachtwerking
F_radiaal_pers1 = askracht_tandriem # [N]
F_tangentieël_pers1 = T_uitgaandeas / (d_as_pers1 / 2) # [N]
F_totaal_pers1 = np.sqrt(F_radiaal_pers1**2 + F_tangentieël_pers1**2) # [N]
print("")
print("INWERKENDE KRACHTEN PERSPASSING 1")
print(f"radiale kracht op de as: {F_radiaal_pers1:.2f} N")
print(f"tangentiële kracht op de as: {F_tangentieël_pers1:.2f} N")
print(f"totale kracht op de as: {F_totaal_pers1:.2f} N")

# beschouwing slipkracht
S_slip_pers1 = 1.75 # gemiddelde waarde tussen 1.5 en 2.0 wat duidt op een gemiddeld belastingsgeval
F_equivalent_pers1 = K_A_uitgaandeas * F_tangentieël_pers1 # [N]
F_s_pers = S_slip_pers1 * F_equivalent_pers1 # [N]
print("")
print("SLIPKRACHT PERSPASSING 1")
print(f"slipkracht voor perspassing 1: {F_s_pers:.2f} N")

# wrijvingscoëfficient
mu_pers1 = np.array([0.18, 0.20]) # interval volgens tabel 12-6a
print("")
print("WRIJVINGSCOËFFICIËNT PERSPASSING 1")
print(f"interval wrijvingscoëfficiënt voor perspassing 1: [{mu_pers1[0]:.2f}, {mu_pers1[1]:.2f}]")

# minimaal vereiste vlakdruk
p_min_pers1 = F_s_pers / (mu_pers1 * oppervlakte_contact_pers1) # [Pa]
print("")
print("MINIMAAL VEREISTE VLAKDRUK PERSPASSING 1")
print(f"interval minimaal vereiste vlakdruk voor perspassing 1: [{p_min_pers1[0]:.2f}, {p_min_pers1[1]:.2f}] Pa")

# gegevens voor minimale perspassing
E_naaf_pers1 = 210e9 # [Pa] # volgens tabel 1-1
E_as_pers1 = 210e9 # [Pa] # volgens tabel 1-1
Q_U_pers1 = d_as_pers1 / steekcirkeldiameter_tandriem # [1] # diameterverhouding naaf
Q_I_pers1 = 0.0 # [1] # diameterverhouding as (massief dus nul)
v_U_pers1 = 0.3 # [1] # volgens tabel 12-6b
v_I_pers1 = 0.3 # [1] # volgens tabel 12-6b

# hulpvariabele K
K_pers1 = ((E_naaf_pers1 / E_as_pers1) * ((1 + Q_I_pers1**2) / (1 - Q_I_pers1**2) - v_I_pers1) + ((1 + Q_U_pers1**2) / (1 - Q_U_pers1**2)) + v_U_pers1) # [Pa]

# minimale persmaat
Z_min_pers1 = ((p_min_pers1 * d_as_pers1) / E_naaf_pers1) * K_pers1 # [m]
print("")
print("MINIMALE PERSMAAT PERSPASSING 1")
print(f"interval minimale persmaat voor perspassing 1: [{Z_min_pers1[0]:.4e}, {Z_min_pers1[1]:.4e}] m")

# ruwheden as en naaf volgens tabel 2-11
R_z_Ui_pers1 = np.array([0.4e-6, 25.0e-6]) # [m] # ruwheid naaf voor ruimen
R_z_Iu_pers1 = np.array([0.16e-6, 25.0e-6]) # [m] # ruwheid as voor rondslijpen

# ruwheidsvermindering
G_pers1 = 0.8 * (R_z_Ui_pers1 + R_z_Iu_pers1) # [m]
print("")
print("RUWHEIDSVERMINDERING PERSPASSING 1")
print(f"interval ruwheidsvermindering voor perspassing 1: [{G_pers1[0]:.4e}, {G_pers1[1]:.4e}] m")

# kleinste toelaatbare overmaat
S_nmin_pers1 = Z_min_pers1 + G_pers1 # [m]
print("")
print("KLEINSTE TOELAATBARE OVERMAAT PERSPASSING 1")
print(f"interval kleinste toelaatbare overmaat voor perspassing 1: [{S_nmin_pers1[0]:.4e}, {S_nmin_pers1[1]:.4e}] m")

# rekgrenzen as en naaf volgens tabel 1-1
ReU_pers1 = 295e6 # [Pa] # rekgrens naaf (materiaal tandriemschijf: E295)
ReI_pers1 = 550e6 # [Pa] # rekgrens as (materiaal as: 38Cr2)

# veiligheidsfactor tegen plastische vervorming
S_pU_pers1 = np.array([1.0, 1.3]) # [1] # voor ductiel materiaal
S_pI_pers1 = np.array([1.0, 1.3]) # [1] # voor ductiel materiaal

# maximale vlakdruk
p_max_buitendeel_pers1 = (ReU_pers1 / S_pU_pers1) * ((1 - Q_U_pers1**2) / np.sqrt(3)) # [Pa]
print("")
print("MAXIMALE VLAKDRUK PERSPASSING 1")
print(f"interval maximale vlakdruk voor buitendeel perspassing 1 (opgelet: kleiner of gelijk aan): [{p_max_buitendeel_pers1[0]:.2e}, {p_max_buitendeel_pers1[1]:.2e}] Pa")
p_max_binnendeel_pers1 = (ReI_pers1 / S_pI_pers1) * (2 / np.sqrt(3)) # [Pa]
print(f"interval maximale vlakdruk voor binnendeel perspassing 1 (opgelet: kleiner of gelijk aan): [{p_max_binnendeel_pers1[0]:.2e}, {p_max_binnendeel_pers1[1]:.2e}] Pa")

# maximale persmaat
Z_max_pers1 = (p_max_buitendeel_pers1 * d_as_pers1 / E_naaf_pers1) * K_pers1 # [m]
print("")
print("MAXIMALE PERSMAAT PERSPASSING 1")
print(f"interval maximale persmaat voor perspassing 1: [{Z_max_pers1[0]:.4e}, {Z_max_pers1[1]:.4e}] m")

# grootste toelaatbare overmaat
S_nmax_pers1 = Z_max_pers1 + G_pers1 # [m]
print("")
print("GROOTSTE TOELAATBARE OVERMAAT PERSPASSING 1")
print(f"interval grootste toelaatbare overmaat voor perspassing 1: [{S_nmax_pers1[0]:.4e}, {S_nmax_pers1[1]:.4e}] m")

# passingstolerantie
tolerantie_pers1 = S_nmax_pers1 - S_nmin_pers1 # [m]
print("")
print("PASSINGSTOLERANTIE PERSPASSING 1")
print(f"interval passingstolerantie voor perspassing 1: [{tolerantie_pers1[0]:.4e}, {tolerantie_pers1[1]:.4e}] m")

PERSPASSING 1: TANDRIEMSCHIJF 2 - UITGAANDE AS

BEPALING INTERVAL NAAFLENGTE (TANDRIEMSCHIJF 2)
interval naaflengte voor tandriemschijf 2: [0.0360, 0.0450] m
praktische naaflengte voor tandriemschijf 2: 0.0360 m

CONTACTOPPERVLAKTE PERSPASSING 1
contactoppervlakte tussen tandriemschijf 2 en uitgaande as: 0.0051 m^2

INWERKENDE KRACHTEN PERSPASSING 1
radiale kracht op de as: 147.95 N
tangentiële kracht op de as: 475.69 N
totale kracht op de as: 498.17 N

SLIPKRACHT PERSPASSING 1
slipkracht voor perspassing 1: 1248.70 N

WRIJVINGSCOËFFICIËNT PERSPASSING 1
interval wrijvingscoëfficiënt voor perspassing 1: [0.18, 0.20]

MINIMAAL VEREISTE VLAKDRUK PERSPASSING 1
interval minimaal vereiste vlakdruk voor perspassing 1: [1363076.98, 1226769.28] Pa

MINIMALE PERSMAAT PERSPASSING 1
interval minimale persmaat voor perspassing 1: [6.0570e-07, 5.4513e-07] m

RUWHEIDSVERMINDERING PERSPASSING 1
interval ruwheidsvermindering voor perspassing 1: [4.4800e-07, 4.0000e-05] m

KLEINSTE TOELAATBARE OVERMAAT 

## i. Berekening dikte achterplaat

In [10]:
# gegevens
M_max_montageplaatM1 = 500 # [N*m]
R_e_staal = 295e6 # [Pa] # rekgrens E295 staal

# berekening vereiste plaatdikte achterplaat
breedte_achterplaat = 0.465 # [m] # uit geometrie
S_veiligheid_plaat = 4.0 # [1] # veiligheidsfactor voor plaat
vereiste_dikte_achterplaat_pl = S_veiligheid_plaat * np.sqrt((6 * M_max_montageplaatM1) / (breedte_achterplaat * R_e_staal)) # [m] # vereenvoudigde formule
print("")
print("BEPALING VEREISTE PLAATDIKTE ACHTERPLAAT OP M1-MONTAGEPLAAT")
print(f"vereiste plaatdikte achterplaat op M1-montageplaat om plastische vervorming te vermijden: {vereiste_dikte_achterplaat_pl:.4e} m")


BEPALING VEREISTE PLAATDIKTE ACHTERPLAAT OP M1-MONTAGEPLAAT
vereiste plaatdikte achterplaat op M1-montageplaat om plastische vervorming te vermijden: 1.8706e-02 m
